1. ESTRUCTURA GENERAL

In [1]:
import numpy as np
from scipy import stats
import pandas as pd
# ======================================
# VARIABLES GLOBALES
# ======================================
reloj = 0

FEL = []

cola = []

estado_servidor = 0     # 0 = libre, 1 = ocupado

clientes_atendidos = 0

num_clientes_objetivo = 1000

cliente_id = 0

# Estadísticas

esperas = []
tiempos_sistema = []
area_cola = 0
area_sistema = 0
area_servidor = 0
ultimo_evento = 0

# Para recordar cuándo llegó cada cliente

tiempos_llegada = {}

# Para recordar cuándo inició servicio cada cliente

inicio_servicio = {}

# Resultados de las 100 corridas

Wq_corridas = []

W_corridas = []

rho_corridas = []

tiempo_final_corridas = []

clientes_sin_espera_corridas = []

Lq_corridas = []
L_corridas = []
rho_real_corridas = []

2. GENERADORES WEIBULL 

In [2]:
# ======================================
# PARÁMETROS WEIBULL
# SE UTILIZARAN LOS PARAMETROS OBTENIDOS EN LA ETAPA 4
# ======================================

#llegadas
c_l = 0.7159872574456299
loc_l = 20.999999999999996
scale_l = 69.62735299085502

#servicios
c_s = 1.9845191143718948
loc_s = 22.02195274464045
scale_s = 117.9238832687666

def generar_interarribo():
    return stats.weibull_min.rvs(
        c_l,
        loc=loc_l,
        scale=scale_l
    )


def generar_servicio():

    return stats.weibull_min.rvs(
        c_s,
        loc=loc_s,
        scale=scale_s
    )

2.1 VALIDACION DE LOS DATOS GENERADOS CON CHI CUADRADO 

In [3]:
import numpy as np
from scipy import stats

def chi_square_test_safe(data, dist, params, bins=10):

    obs, edges = np.histogram(data, bins=bins)

    expected = []

    for i in range(len(edges)-1):
        p = dist.cdf(edges[i+1], *params) - dist.cdf(edges[i], *params)
        expected.append(len(data) * p)

    expected = np.array(expected)

    mask = expected >= 5

    obs = obs[mask]
    expected = expected[mask]

    if len(obs) < 2:
        return None, None

    # Ajustar para que ambas sumas coincidan
    expected = expected * (obs.sum() / expected.sum())

    chi2, p = stats.chisquare(obs, expected)

    return chi2, p


interarribos = [generar_interarribo() for _ in range(1000)]

chi2, p = chi_square_test_safe(
    interarribos,
    stats.weibull_min,
    (c_l, loc_l, scale_l)
)

print("\nVALIDACIÓN BONDAD DE AJUSTE - INTERARRIBOS- DISTRIBUCION WEIBULL")
print("=" * 50)
print(f"Chi-cuadrado: {chi2:.3f}")
print(f"p-value     : {p:.5f}")

if p > 0.05:
    print("Conclusión: No se rechaza H0 → ajuste adecuado")
else:
    print("Conclusión: Se rechaza H0 → ajuste no adecuado")
    

servicios = [generar_servicio() for _ in range(1000)]

chi2, p = chi_square_test_safe(
    servicios,
    stats.weibull_min,
    (c_s, loc_s, scale_s)
)

print("\nVALIDACIÓN BONDAD DE AJUSTE - SERVICIOS- DISTRIBUCION WEIBULL")
print("=" * 50)
print(f"Chi-cuadrado: {chi2:.3f}")
print(f"p-value     : {p:.5f}")

if p > 0.05:
    print("Conclusión: No se rechaza H0 → ajuste adecuado")
else:
    print("Conclusión: Se rechaza H0 → ajuste no adecuado")


VALIDACIÓN BONDAD DE AJUSTE - INTERARRIBOS- DISTRIBUCION WEIBULL
Chi-cuadrado: 2.088
p-value     : 0.55436
Conclusión: No se rechaza H0 → ajuste adecuado

VALIDACIÓN BONDAD DE AJUSTE - SERVICIOS- DISTRIBUCION WEIBULL
Chi-cuadrado: 5.213
p-value     : 0.63393
Conclusión: No se rechaza H0 → ajuste adecuado


3. SIMULACION COMPLETA
3.1  INICIALIZAR

In [4]:
def inicializar():

    global reloj
    global FEL
    global cola
    global estado_servidor
    global clientes_atendidos
    global cliente_id
    global esperas
    global tiempos_sistema
    global tiempos_llegada
    global inicio_servicio
    global area_cola
    global area_sistema
    global area_servidor
    global ultimo_evento

    reloj = 0

    FEL = []

    cola = []

    estado_servidor = 0

    clientes_atendidos = 0

    cliente_id = 0
    
    area_cola = 0
    area_sistema = 0
    area_servidor = 0
    ultimo_evento = 0
        
    esperas = []

    tiempos_sistema = []

    tiempos_llegada = {}

    inicio_servicio = {}

    # Programar primera llegada

    tiempo_primera_llegada = generar_interarribo()

    FEL.append({
        "tipo": "LLEGADA",
        "tiempo": tiempo_primera_llegada,
        "cliente": cliente_id
    })

    cliente_id += 1

3.2 TIMING

In [5]:
def timing():

    global reloj
    global area_cola
    global area_sistema
    global area_servidor
    global ultimo_evento

    FEL.sort(key=lambda e: e["tiempo"])
    evento = FEL.pop(0)

    # Δt
    tiempo_actual = evento["tiempo"]
    delta = tiempo_actual - ultimo_evento

    # estado actual
    q_len = len(cola)
    in_system = q_len + estado_servidor

    # acumulación áreas (tipo Etapa 3)
    area_cola += q_len * delta
    area_sistema += in_system * delta
    area_servidor += estado_servidor * delta

    ultimo_evento = tiempo_actual
    reloj = tiempo_actual

    return evento

3.3 LLEGADAS

In [6]:
def llegada(evento):

    global cliente_id
    global estado_servidor

    id_cliente = evento["cliente"]

    tiempos_llegada[id_cliente] = reloj

    # Programar próxima llegada

    siguiente_llegada = reloj + generar_interarribo()

    FEL.append({
        "tipo": "LLEGADA",
        "tiempo": siguiente_llegada,
        "cliente": cliente_id
    })

    cliente_id += 1

    # Servidor libre

    if estado_servidor == 0:

        estado_servidor = 1
        
        inicio_servicio[id_cliente] = reloj
        
        servicio = generar_servicio()

        FEL.append({
            "tipo": "SALIDA",
            "tiempo": reloj + servicio,
            "cliente": id_cliente
        })

    else:

        cola.append(id_cliente)

3.4 SALIDAS

In [7]:
def salida(evento):

    global clientes_atendidos
    global estado_servidor

    id_cliente = evento["cliente"]

    clientes_atendidos += 1

    llegada_cliente = tiempos_llegada[id_cliente]

    inicio_cliente = inicio_servicio[id_cliente]

    espera = inicio_cliente - llegada_cliente

    tiempo_sistema = reloj - llegada_cliente

    esperas.append(espera)

    tiempos_sistema.append(tiempo_sistema)

    if len(cola) > 0:

        siguiente_cliente = cola.pop(0)

        inicio_servicio[siguiente_cliente] = reloj

        servicio = generar_servicio()

        FEL.append({
            "tipo": "SALIDA",
            "tiempo": reloj + servicio,
            "cliente": siguiente_cliente
        })

    else:

        estado_servidor = 0

3.5 REPORTE

In [8]:

def obtener_metricas():

    media_llegadas = stats.weibull_min.mean(
        c_l,
        loc=loc_l,
        scale=scale_l
    )

    media_servicios = stats.weibull_min.mean(
        c_s,
        loc=loc_s,
        scale=scale_s
    )

    lambda_teorico = 1 / media_llegadas
    mu_teorico = 1 / media_servicios

    rho_teorico = lambda_teorico / mu_teorico
    tiempo = max(1e-9, reloj)
    
    Lq_teorico = lambda_teorico * np.mean(esperas)
    L_teorico = lambda_teorico * np.mean(tiempos_sistema)
    return {
        "Wq": np.mean(esperas),
        "W": np.mean(tiempos_sistema),
        "Lq": area_cola / tiempo,
        "L": area_sistema / tiempo,
        "rho_real": area_servidor / max(1e-9, reloj),
        "rho_teorico": rho_teorico,
        "Lq_teorico": Lq_teorico,
        "L_teorico": L_teorico,
        "espera_min": np.min(esperas),
        "espera_max": np.max(esperas),
        "sin_espera": sum(1 for x in esperas if x == 0),
        "tiempo_final": reloj
        
    }

4. PROGRAMA PRINCIPAL (MAIN)

In [9]:
for corrida in range(100):

    inicializar()

    while clientes_atendidos < num_clientes_objetivo:

        evento = timing()

        if evento["tipo"] == "LLEGADA":

            llegada(evento)

        else:

            salida(evento)

    resultados = obtener_metricas()
    
    Wq_corridas.append(resultados["Wq"])

    W_corridas.append(resultados["W"])

    rho_corridas.append(resultados["rho_teorico"])

    tiempo_final_corridas.append(reloj)

    Lq_corridas.append(resultados["Lq"])
    L_corridas.append(resultados["L"])
    rho_real_corridas.append(resultados["rho_real"])
    clientes_sin_espera_corridas.append(
        sum(1 for x in esperas if x == 0)
    )
    #reporte()

tabla = pd.DataFrame({
    "Corrida": range(1, 101),
    "Wq": Wq_corridas,
    "W": W_corridas,
    "Lq": Lq_corridas,
    "L": L_corridas,
    "Rho_real": rho_real_corridas,
    "Rho_teorico": rho_corridas,
    "Tiempo_Final": tiempo_final_corridas,
    "Sin_Espera": clientes_sin_espera_corridas
})

print("\nESTADISTICAS GENERALES")
print(tabla.describe())

tabla.to_excel("Resultados_100_Corridas.xlsx", index=False)

print("\nTABLA DE RESULTADOS")
print("=" * 50)
print("Los resultados completos han sido exportados correctamente.")
print("Archivo generado: Resultados_100_Corridas.xlsx")
print("Ubicación: directorio de ejecución del programa")
print("=" * 50)
pd.set_option('display.width', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_rows', None)
print(tabla)
print("\n====================================")
print("RESUMEN 100 CORRIDAS")
print("====================================")
print(f"Wq promedio: {np.mean(Wq_corridas):.3f}")
print(f"W promedio: {np.mean(W_corridas):.3f}")
print(f"Lq promedio: {np.mean(Lq_corridas):.3f}")
print(f"L promedio: {np.mean(L_corridas):.3f}")
print(f"ρ real promedio: {np.mean(rho_real_corridas):.3f}")
print(f"ρ teórico promedio: {np.mean(rho_corridas):.3f}")




ESTADISTICAS GENERALES
          Corrida            Wq             W          Lq           L  \
count  100.000000    100.000000    100.000000  100.000000  100.000000   
mean    50.500000   9603.326869   9729.507557   89.817114   90.812044   
std     29.011492   2325.869605   2326.599853   23.815358   23.815562   
min      1.000000   4776.468943   4897.055446   43.155378   44.153686   
25%     25.750000   7758.356752   7884.652639   71.041568   72.039910   
50%     50.500000   9406.920199   9534.793669   89.505677   90.488599   
75%     75.250000  11207.827905  11334.299970  106.319690  107.316544   
max    100.000000  16467.086330  16593.154899  166.466424  167.463785   

         Rho_real  Rho_teorico   Tiempo_Final  Sin_Espera  
count  100.000000   100.000000     100.000000  100.000000  
mean     0.994930     1.179339  126826.258027    4.440000  
std      0.005220     0.000000    1954.592278    3.666171  
min      0.968404     1.179339  120790.793605    1.000000  
25%      0.992003 

5. PRUEBAS DE AUTOCORRELACION

In [10]:
import numpy as np

# =========================================================
# 1. FUNCIÓN AUTOCORRELACIÓN (LAG 1)
# =========================================================
def autocorrelacion_lag1(x):
    x = np.array(x)
    return np.corrcoef(x[:-1], x[1:])[0, 1]


print("\nAUTOCORRELACIÓN ENTRE CORRIDAS (ANÁLISIS BASE)")
print("================================================")

wq_ac = autocorrelacion_lag1(Wq_corridas)
w_ac  = autocorrelacion_lag1(W_corridas)
lq_ac = autocorrelacion_lag1(Lq_corridas)
l_ac  = autocorrelacion_lag1(L_corridas)

print(f"Wq lag1: {wq_ac:.4f}")
print(f"W  lag1: {w_ac:.4f}")
print(f"Lq lag1: {lq_ac:.4f}")
print(f"L  lag1: {l_ac:.4f}")


# =========================================================
# 2. INTERPRETACIÓN CORRECTA
# =========================================================
print("\nINTERPRETACIÓN")
print("===============")
print("- Valores cercanos a 0 indican baja o nula autocorrelación")
print("- Valores cercanos a ±1 indican fuerte dependencia")
print("\nConclusión: se evalúa si existe dependencia entre corridas antes de aplicar técnicas de estabilización.")


# =========================================================
# 3. MÉTODO BATCH MEANS
# =========================================================
def batch_means(data, m=10):
    data = np.array(data)

    k = len(data) // m          # número de batches
    data = data[:k*m]           # recortar sobrantes

    batches = data.reshape(k, m).mean(axis=1)

    return batches


# Aplicar Batch Means a las métricas
Wq_batch = batch_means(Wq_corridas, m=10)
W_batch  = batch_means(W_corridas, m=10)
Lq_batch = batch_means(Lq_corridas, m=10)
L_batch  = batch_means(L_corridas, m=10)


# =========================================================
# 4. AUTOCORRELACIÓN EN DATOS AGRUPADOS
# =========================================================
print("\nAUTOCORRELACIÓN (DATOS AGRUPADOS - BATCH MEANS)")
print("================================================")

print(f"Wq batch lag1: {autocorrelacion_lag1(Wq_batch):.4f}")
print(f"W  batch lag1: {autocorrelacion_lag1(W_batch):.4f}")
print(f"Lq batch lag1: {autocorrelacion_lag1(Lq_batch):.4f}")
print(f"L  batch lag1: {autocorrelacion_lag1(L_batch):.4f}")


# =========================================================
# 5. CONCLUSIÓN CORRECTA (IMPORTANTE PARA EL INFORME)
# =========================================================
print("\nCONCLUSIÓN")
print("===========")
print("Las corridas originales presentan baja autocorrelación,")
print("por lo que el sistema puede considerarse aproximadamente independiente.")

print("\nEl método Batch Means se aplica como técnica de agregación")
print("para mejorar la estabilidad de los estimadores y permitir")
print("la construcción de intervalos de confianza en estado estable.")


AUTOCORRELACIÓN ENTRE CORRIDAS (ANÁLISIS BASE)
Wq lag1: -0.0212
W  lag1: -0.0213
Lq lag1: -0.0257
L  lag1: -0.0256

INTERPRETACIÓN
- Valores cercanos a 0 indican baja o nula autocorrelación
- Valores cercanos a ±1 indican fuerte dependencia

Conclusión: se evalúa si existe dependencia entre corridas antes de aplicar técnicas de estabilización.

AUTOCORRELACIÓN (DATOS AGRUPADOS - BATCH MEANS)
Wq batch lag1: -0.4417
W  batch lag1: -0.4416
Lq batch lag1: -0.3393
L  batch lag1: -0.3394

CONCLUSIÓN
Las corridas originales presentan baja autocorrelación,
por lo que el sistema puede considerarse aproximadamente independiente.

El método Batch Means se aplica como técnica de agregación
para mejorar la estabilidad de los estimadores y permitir
la construcción de intervalos de confianza en estado estable.


6. INTERVALOS DE CONFIANZA 

In [11]:
import numpy as np
from scipy import stats
# USAMOS TSTUDENT
def intervalo_confianza(data, alpha=0.05):
    data = np.array(data)
    
    n = len(data)
    media = np.mean(data)
    varianza = np.var(data, ddof=1)
    std = np.sqrt(varianza)

    t = stats.t.ppf(1 - alpha/2, df=n-1)

    error = t * (std / np.sqrt(n))

    return media, varianza, (media - error, media + error)



print("\nINTERVALOS DE CONFIANZA AL 95% (BATCH MEANS)")
print("=============================================")

# Wq
media_wq, var_wq, ci_wq = intervalo_confianza(Wq_batch)
print("\nWq:")
print(f"Media = {media_wq:.3f}")
print(f"Varianza = {var_wq:.3f}")
print(f"IC 95% = {ci_wq}")

# W
media_w, var_w, ci_w = intervalo_confianza(W_batch)
print("\nW:")
print(f"Media = {media_w:.3f}")
print(f"Varianza = {var_w:.3f}")
print(f"IC 95% = {ci_w}")

# Lq
media_lq, var_lq, ci_lq = intervalo_confianza(Lq_batch)
print("\nLq:")
print(f"Media = {media_lq:.3f}")
print(f"Varianza = {var_lq:.3f}")
print(f"IC 95% = {ci_lq}")

# L
media_l, var_l, ci_l = intervalo_confianza(L_batch)
print("\nL:")
print(f"Media = {media_l:.3f}")
print(f"Varianza = {var_l:.3f}")
print(f"IC 95% = {ci_l}")


INTERVALOS DE CONFIANZA AL 95% (BATCH MEANS)

Wq:
Media = 9603.327
Varianza = 547131.069
IC 95% = (9074.189464193016, 10132.46427281601)

W:
Media = 9729.508
Varianza = 547794.051
IC 95% = (9200.049660214383, 10258.965453137156)

Lq:
Media = 89.817
Varianza = 48.203
IC 95% = (84.85051269338598, 94.78371562982497)

L:
Media = 90.812
Varianza = 48.209
IC 95% = (85.84514539011153, 95.7789429115605)
